# Task 3: Broadcast Join Experiment (Query 1)

- This experiment compares normal joins vs. broadcast joins using the silver taxi-trips and silver taxi-zone Delta tables for Query 1.
- It verifies that forcing a broadcast join reduces shuffle operations and improves performance.

## Connect to Spark

In [ ]:
import sys
from pyspark.sql import SparkSession

JOB_PATH = '/opt/spark/jobs'
if JOB_PATH not in sys.path:
    sys.path.insert(0, JOB_PATH)

from analytics.benchmarking import measure_registered_query, rows_match
from analytics.runner import execute_registered_query
from analytics.source_tables import register_broadcast_join_sources

spark = (
    SparkSession.builder
    .master('spark://spark-master:7077')
    .appName('Assignment2-BroadcastJoin-Query1')
    .getOrCreate()
)
spark.conf.set('spark.sql.session.timeZone', 'UTC')

## Register the original Delta tables

In [ ]:
spark.catalog.clearCache()
register_broadcast_join_sources(spark, register_zones=True, register_air_quality=False)
print('Registered Silver taxi trips and taxi zone views.')

## Isolate the join strategy

- Automatic broadcasting is disabled to prevent Spark from optimizing the 'normal' query.
- AQE is disabled to ensure the plan remains static during the experiment.

In [ ]:
JOIN_CONFIGS = {
    'spark.sql.adaptive.enabled': 'false',
    'spark.sql.autoBroadcastJoinThreshold': '-1',
}
for key, value in JOIN_CONFIGS.items():
    spark.conf.set(key, value)

JOIN_CONFIGS

## Measure normal and broadcast joins

In [ ]:
NORMAL_QUERY = 'query_01_broadcast_normal'
BROADCAST_QUERY = 'query_01_broadcast_forced'
REPETITIONS = 3

normal_results = measure_registered_query(spark, NORMAL_QUERY, REPETITIONS)
broadcast_results = measure_registered_query(spark, BROADCAST_QUERY, REPETITIONS)

assert rows_match(
    normal_results['rows'],
    broadcast_results['rows'],
), 'Normal and broadcast results differ.'

print('Normal and broadcast results match within float tolerance.')

## Normal-join physical plan
Expect `SortMergeJoin` here.

In [ ]:
execute_registered_query(spark, NORMAL_QUERY).explain('formatted')

## Broadcast-join physical plan
Expect `BroadcastHashJoin` and `BroadcastExchange` here.

In [ ]:
execute_registered_query(spark, BROADCAST_QUERY).explain('formatted')

## Benchmark summary

In [ ]:
summary = {
    'normal_samples_seconds': [round(value, 3) for value in normal_results['samples_seconds']],
    'normal_median_seconds': round(normal_results['median_seconds'], 3),
    'broadcast_samples_seconds': [round(value, 3) for value in broadcast_results['samples_seconds']],
    'broadcast_median_seconds': round(broadcast_results['median_seconds'], 3),
    'broadcast_speedup_factor': round(normal_results['median_seconds'] / broadcast_results['median_seconds'], 2),
}
summary

## Cleanup

In [ ]:
spark.catalog.clearCache()
spark.stop()